In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from sklearn.linear_model import LinearRegression
import numpy as np

# Cargar datos
df = pd.read_csv('../data/turistas.csv')
df_agg = df.groupby('anio')['visitantes'].sum().reset_index()
df_agg = df_agg.rename(columns={'anio': 'ds', 'visitantes': 'y'})
df_agg['ds'] = pd.to_datetime(df_agg['ds'], format='%Y')
df_agg

In [ ]:
# --- Modelo Prophet ---
prophet_model = Prophet()
prophet_model.fit(df_agg)
future = prophet_model.make_future_dataframe(periods=5, freq='Y')
forecast_prophet = prophet_model.predict(future)

fig1 = prophet_model.plot(forecast_prophet)
plt.title('Proyección con Prophet (5 años)')
plt.show()

In [ ]:
# --- Modelo ARIMA ---
arima_data = df_agg.set_index('ds')['y']
arima_model = ARIMA(arima_data, order=(1,1,1))
arima_fit = arima_model.fit()
forecast_arima = arima_fit.forecast(steps=5)

plt.figure(figsize=(8,5))
plt.plot(arima_data.index, arima_data.values, label='Histórico')
plt.plot(pd.date_range(arima_data.index[-1], periods=6, freq='Y')[1:], forecast_arima, label='ARIMA Forecast')
plt.legend()
plt.title('Proyección con ARIMA (5 años)')
plt.show()

In [ ]:
# --- Regresión Lineal ---
X = np.array(range(len(df_agg))).reshape(-1,1)
y = df_agg['y'].values
model_lr = LinearRegression().fit(X, y)

# Predicciones futuras
future_X = np.array(range(len(df_agg)+5)).reshape(-1,1)
y_pred = model_lr.predict(future_X)

plt.figure(figsize=(8,5))
plt.plot(df_agg['ds'], y, label='Histórico')
plt.plot(pd.date_range(df_agg['ds'].iloc[0], periods=len(future_X), freq='Y'), y_pred, label='Regresión Lineal')
plt.legend()
plt.title('Proyección con Regresión Lineal (5 años)')
plt.show()